# EXP 3 G Sampling Experiment

Reliability check for `base`, `cw`, `smote`, `smote+cw`, `smotenc`, and `smote+nc` using the same EXP3 preprocessing bundle used by the app.
Focus: detect physiologically/medically impossible synthetic patients.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

In [ ]:
ROOT = Path.cwd()
BUNDLE_CANDIDATES = [
    ROOT / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
    ROOT / 'V2.2.1.1' / 'exp3_logreg_cat' / 'models' / 'best_model_bundle.joblib',
]
DATA_CANDIDATES = [
    ROOT / 'merged_clinical_leftjoin.csv',
    ROOT / 'merged_clinical_dietary_anthro_leftjoin.csv',
    ROOT / 'merged_clinical_dietary_leftjoin.csv',
    ROOT / 'V2.2.1.1' / 'merged_clinical_leftjoin.csv',
    ROOT / 'V2.2.1.1' / 'merged_clinical_dietary_anthro_leftjoin.csv',
]

bundle_path = next((p for p in BUNDLE_CANDIDATES if p.exists()), None)
data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if bundle_path is None:
    raise FileNotFoundError(f'Bundle not found. Checked: {BUNDLE_CANDIDATES}')
if data_path is None:
    raise FileNotFoundError(f'Dataset not found. Checked: {DATA_CANDIDATES}')

bundle = joblib.load(bundle_path)
df = pd.read_csv(data_path)

target_candidates = ['hypertension', 'htn', 'target', 'label', 'outcome']
target_col = next((c for c in df.columns if c.lower() in target_candidates), None)
if target_col is None:
    raise ValueError(f'No target column found from {target_candidates}')

input_features = list(bundle['input_feature_names'])
missing_feats = [c for c in input_features if c not in df.columns]
if missing_feats:
    raise ValueError(f'Missing required input features in dataset: {missing_feats[:12]}')

X = df[input_features].copy()
y = df[target_col].astype(int).copy()
print('Bundle:', bundle_path)
print('Dataset:', data_path)
print('Rows:', len(df), '| Positives:', int(y.sum()), '| Features:', len(input_features))

In [ ]:
# Same preprocessing foundations as EXP3 copy: KNN impute numerics + categorical impute.
num_cols = list(bundle['num_cols_full'])
cat_cols = list(bundle['cat_cols'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

if num_cols:
    Xn_train = pd.DataFrame(bundle['knn_imputer'].transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
else:
    Xn_train = pd.DataFrame(index=X_train.index)

if cat_cols and bundle.get('cat_imputer') is not None:
    Xc_train = pd.DataFrame(bundle['cat_imputer'].transform(X_train[cat_cols]), columns=cat_cols, index=X_train.index).astype(str)
else:
    Xc_train = pd.DataFrame(index=X_train.index)

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
if cat_cols:
    Xc_train_enc = pd.DataFrame(enc.fit_transform(Xc_train), columns=cat_cols, index=X_train.index)
else:
    Xc_train_enc = pd.DataFrame(index=X_train.index)

X_train_sampling = pd.concat([Xn_train, Xc_train_enc], axis=1)
feature_cols_sampling = list(X_train_sampling.columns)
cat_idx = [feature_cols_sampling.index(c) for c in cat_cols if c in feature_cols_sampling]

print('Sampling matrix shape:', X_train_sampling.shape, '| cat features for SMOTENC:', len(cat_idx))

In [ ]:
def impossible_flags(df_sample: pd.DataFrame) -> pd.Series:
    f = pd.Series(False, index=df_sample.index)

    if 'age' in df_sample.columns:
        f |= (df_sample['age'] < 0) | (df_sample['age'] > 120)
    if 'sex' in df_sample.columns:
        f |= ~df_sample['sex'].round().isin([1, 2])
    if 'height' in df_sample.columns:
        f |= (df_sample['height'] <= 0) | (df_sample['height'] > 260)
    if 'weight' in df_sample.columns:
        f |= (df_sample['weight'] <= 0) | (df_sample['weight'] > 350)
    if 'waist' in df_sample.columns:
        f |= (df_sample['waist'] <= 0) | (df_sample['waist'] > 200)
    if 'hip' in df_sample.columns:
        f |= (df_sample['hip'] <= 0) | (df_sample['hip'] > 220)
    if 'BMI' in df_sample.columns:
        f |= (df_sample['BMI'] < 10) | (df_sample['BMI'] > 80)
    if 'bmi' in df_sample.columns:
        f |= (df_sample['bmi'] < 10) | (df_sample['bmi'] > 80)
    if 'whr' in df_sample.columns:
        f |= (df_sample['whr'] < 0.4) | (df_sample['whr'] > 2.0)

    nonneg_prefixes = ('Total_', 'fg', 'epwt_fg')
    for c in df_sample.columns:
        if c.startswith(nonneg_prefixes):
            f |= (pd.to_numeric(df_sample[c], errors='coerce') < 0)

    return f.fillna(True)

def resample_method(name: str, Xs: pd.DataFrame, ys: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    if name in {'base', 'cw'}:
        return Xs.copy(), ys.copy()
    if name in {'smote', 'smote+cw'}:
        sampler = SMOTE(random_state=42, k_neighbors=5)
        Xr, yr = sampler.fit_resample(Xs, ys)
        return pd.DataFrame(Xr, columns=Xs.columns), pd.Series(yr)
    if name in {'smotenc', 'smote+nc'}:
        sampler = SMOTENC(categorical_features=cat_idx, random_state=42, k_neighbors=5)
        Xr, yr = sampler.fit_resample(Xs, ys)
        return pd.DataFrame(Xr, columns=Xs.columns), pd.Series(yr)
    raise ValueError(name)

In [ ]:
methods = ['base', 'cw', 'smote', 'smote+cw', 'smotenc', 'smote+nc']
summary_rows = []
examples = {}

for m in methods:
    Xr, yr = resample_method(m, X_train_sampling, y_train)
    flags = impossible_flags(Xr)
    n_bad = int(flags.sum())
    n_all = int(len(flags))
    summary_rows.append({
        'method': m,
        'rows': n_all,
        'positive_rate': float(np.mean(yr)),
        'impossible_rows': n_bad,
        'impossible_pct': (100.0 * n_bad / max(n_all, 1)),
    })
    if n_bad > 0:
        examples[m] = Xr.loc[flags].head(5).copy()

summary_df = pd.DataFrame(summary_rows).sort_values(['impossible_pct', 'method']).reset_index(drop=True)
summary_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.bar(summary_df['method'], summary_df['impossible_pct'])
plt.ylabel('Impossible synthetic patients (%)')
plt.xlabel('Sampling strategy')
plt.title('Physiological/Medical Plausibility Check per Sampling Method')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Inspect first impossible examples per method (if any).
for m in methods:
    print('\n===', m, '===')
    if m not in examples:
        print('No impossible rows detected.')
    else:
        display(examples[m])